# WhisperX on SageMaker -- Real-Time Speech-to-Text

Deploy the **WhisperX** Deep Learning Container to a SageMaker real-time endpoint, warm it up, and transcribe a multi-speaker clip into a **color-coded, speaker-labeled** transcript -- with a live throughput number.

## What this shows

1. **~70x-realtime batched transcription** -- WhisperX runs `large-v2` with batched inference, transcribing far faster than the audio plays.
2. **Word-level timestamps via wav2vec2 forced alignment** -- every word gets precise start/end times, not just per-segment.
3. **Speaker diarization** -- who spoke when, so each line is labeled `SPEAKER_00`, `SPEAKER_01`, ...

## Prerequisites

- A **SageMaker Studio** notebook with an execution role that can `create_model` / `create_endpoint`.
- **Outbound internet** -- the notebook downloads the Apollo clip from Wikimedia Commons and pip-installs `imageio-ffmpeg` if `ffmpeg` isn't present.
- The **built WhisperX image URI** (paste it into `IMAGE_URI` in the config cell).
- **GPU service quota** for the endpoint instance type (`ml.g4dn.xlarge` by default).

> ### ⚠️ CRITICAL -- the `InferenceAmiVersion` pin
> This endpoint **must** set `InferenceAmiVersion = "al2-ami-sagemaker-inference-gpu-3-1"` (CUDA 12 / driver 550).
> **Without this pin the GPU endpoint fails to start with a zero-log `CannotStartContainerError`.** Do not change it.

> ### 💸 Cost / teardown warning
> A GPU real-time endpoint **bills continuously until you delete it.** Run the **final teardown cell** as soon as the demo is over.

In [ ]:
# ============================= CONFIG =============================
REGION = "us-west-2"

# PASTE the WhisperX DLC image URI here (.../whisperx:<tag>).
IMAGE_URI = "763104351884.dkr.ecr.us-west-2.amazonaws.com/whisperx:3.8.6-cu128-amzn2023-sagemaker"  

INSTANCE_TYPE = "ml.g4dn.xlarge"   # T4 16GB. Swap to "ml.g5.xlarge" (A10G) for more GPU headroom.

# REQUIRED -- do not change. CUDA-12 / driver-550 AMI. Without this pin the GPU
# endpoint fails to start with a zero-log CannotStartContainerError.
INFERENCE_AMI_VERSION = "al2-ami-sagemaker-inference-gpu-3-1"

STARTUP_HEALTH_CHECK_TIMEOUT = 900   # seconds the container gets to pass health checks
DEPLOY_TIMEOUT = 1800                # seconds to wait for the endpoint to reach InService
ROLE_ARN = ""                      # fallback used only if get_execution_role() is unavailable
BASE_NAME = "whisperx-demo"
# =================================================================

import boto3, time, json, os, uuid
from botocore.exceptions import ClientError, ReadTimeoutError

sm = boto3.client("sagemaker", region_name=REGION)
smr = boto3.client("sagemaker-runtime", region_name=REGION)

# Resolve the execution role. Inside Studio get_execution_role() just works;
# elsewhere we fall back to the ROLE_ARN config constant.
role = ROLE_ARN
if not role:
      role = None
      # 1) In Studio/notebook the SageMaker SDK returns the full ARN (with path).
      try:
          from sagemaker.session import get_execution_role
          role = get_execution_role()
      except Exception:
          role = None
      # 2) Fallback: STS gives only the role NAME (no path) -> resolve the real ARN via IAM.
      if not role:
          ident = boto3.client("sts").get_caller_identity()["Arn"]
          if ":assumed-role/" in ident:
              role_name = ident.split("assumed-role/")[1].split("/")[0]
              try:
                  role = boto3.client("iam").get_role(RoleName=role_name)["Role"]["Arn"]
              except ClientError as e:
                  # No iam:GetRole permission -> path is NOT recoverable from STS alone.
                  acct = ident.split(":")[4]
                  role = f"arn:aws:iam::{acct}:role/{role_name}"
                  print(f"WARN: iam:GetRole denied ({e.response['Error']['Code']}); "
                        f"using path-less ARN {role}. Set ROLE_ARN explicitly if this role has a path.")
          else:
              role = ident  # already a user/role ARN

# Unique names so a re-run never collides with a leftover endpoint.
suffix = uuid.uuid4().hex[:8]
MODEL_NAME = f"{BASE_NAME}-model-{suffix}"
CONFIG_NAME = f"{BASE_NAME}-config-{suffix}"
ENDPOINT_NAME = f"{BASE_NAME}-ep-{suffix}"

print("Region:   ", REGION)
print("Role:     ", role)
print("Model:    ", MODEL_NAME)
print("Config:   ", CONFIG_NAME)
print("Endpoint: ", ENDPOINT_NAME)
print("Instance: ", INSTANCE_TYPE)

## 1. Deploy the endpoint

We create a **Model** (just the container image -- `large-v2` weights download at runtime, diarization weights are baked in, **no HuggingFace token needed**), an **EndpointConfig** with the GPU AMI pin, and the **Endpoint**.

This takes about **5-10 minutes**. Note the endpoint reports **InService before the model weights finish downloading** -- that first download happens on the first request, which is why we run a **warm-up** call first.

In [ ]:
t0 = time.time()
try:
    # Model = container image only. No ModelDataUrl: large-v2 downloads at runtime
    # and the diarization weights are baked into the image (no HuggingFace token).
    sm.create_model(
        ModelName=MODEL_NAME,
        PrimaryContainer={"Image": IMAGE_URI},
        ExecutionRoleArn=role,
    )

    sm.create_endpoint_config(
        EndpointConfigName=CONFIG_NAME,
        ProductionVariants=[{
            "VariantName": "AllTraffic",
            "ModelName": MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": INSTANCE_TYPE,
            # >>> CRITICAL: without this pin the GPU endpoint fails to start with a
            # >>> zero-log CannotStartContainerError. CUDA-12 / driver-550. DO NOT REMOVE.
            "InferenceAmiVersion": INFERENCE_AMI_VERSION,
            "ContainerStartupHealthCheckTimeoutInSeconds": STARTUP_HEALTH_CHECK_TIMEOUT,
        }],
    )

    sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=CONFIG_NAME)
    print(f"Creating endpoint {ENDPOINT_NAME} ...")

    # Poll until InService; bail loudly on Failed.
    last_status = None
    while True:
        d = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
        status = d["EndpointStatus"]
        elapsed = time.time() - t0
        if status != last_status:
            print(f"  [{elapsed:6.0f}s] {status}")
            last_status = status
        if status == "InService":
            break
        if status == "Failed":
            raise RuntimeError("Endpoint failed: " + d.get("FailureReason", "<no reason>"))
        if elapsed > DEPLOY_TIMEOUT:
            raise TimeoutError(f"Endpoint not InService after {DEPLOY_TIMEOUT}s (last status {status}).")
        time.sleep(15)

    print(f"Endpoint InService in {time.time() - t0:.0f}s.")
except ClientError as e:
    # Most commonly the model/config/endpoint name already exists from a prior run.
    print("Deploy call failed:", e)
    print("If this says a name already exists, re-run the CONFIG cell for a fresh "
          "suffix, or run the teardown cell first.")
    raise

## 2. Get the audio -- Apollo 11 landing (public domain)

The demo clip is NASA's **Apollo 11 Moon-landing** audio from **Wikimedia Commons** (**PD-USGov-NASA**): Aldrin's descent callouts, Armstrong's *"Houston, Tranquility Base here, the Eagle has landed,"* and CAPCOM Charlie Duke's *"we copy you down ... a bunch of guys about to turn blue, we're breathing again."* (A shorter *"one small step"* clip is the fallback.) The next cell downloads it and transcodes to 16 kHz mono WAV.

**Trim knobs -- `APOLLO_START_SEC` / `APOLLO_DUR_SEC`:**

- Leave **both `None`** (the default) to transcribe the **full ~2:34 clip**. Do this **first, as a dry run**: read the color-coded transcript, find where *"the Eagle has landed"* falls, and note the timestamps.
- For the **live run**, set them to a **~45s window** around the landing (e.g. `APOLLO_START_SEC = 95`, `APOLLO_DUR_SEC = 55`, confirmed from your dry-run transcript). This keeps the call well under the 60s real-time cap.

> Full-clip transcription + word alignment + diarization can **approach the 60s real-time cap on `ml.g4dn.xlarge`.** Either **trim** (above) or switch `INSTANCE_TYPE` to **`ml.g5.xlarge`** (A10G) for headroom.

To use **your own** clip instead, set `LOCAL_AUDIO` to a local `.wav` / `.mp3` / `.ogg` path. If `ffmpeg` isn't on `PATH` (common in Studio), the cell pip-installs a static **`imageio-ffmpeg`** binary automatically.

In [ ]:
# ===================== DEMO AUDIO: Apollo 11 (public domain) =====================
# NASA Moon-landing audio from Wikimedia Commons (PD-USGov-NASA). Multi-speaker:
# Aldrin's descent callouts, Armstrong's "Houston, Tranquility Base here, the Eagle
# has landed," and CAPCOM Charlie Duke's "we copy you down ... a bunch of guys about
# to turn blue, we're breathing again."
LOCAL_AUDIO = None   # set to a local audio path (.wav/.mp3/.ogg) to use your own clip

# Trim window for the LIVE run (seconds). Leave BOTH None for the full ~2:34 clip --
# ideal for a first dry run: read the timestamps, find "the Eagle has landed," then
# set these to a ~45s window (e.g. 95 / 55) so the live call stays well under the 60s
# real-time cap. Full clip + diarization can approach that cap on ml.g4dn.xlarge.
APOLLO_START_SEC = None   # e.g. 95
APOLLO_DUR_SEC   = None   # e.g. 55

APOLLO_URL = "https://upload.wikimedia.org/wikipedia/commons/e/e5/Pouso_da_Apollo_11_na_Lua.ogg"
APOLLO_FALLBACK_URL = "https://upload.wikimedia.org/wikipedia/commons/d/dd/Armstrong_Small_Step.ogg"

import sys, shutil, subprocess, urllib.request

def _ffmpeg_exe():
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    # No system ffmpeg (common in Studio) -> pull a static binary.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imageio-ffmpeg"], check=True)
    import imageio_ffmpeg
    return imageio_ffmpeg.get_ffmpeg_exe()

def _download(url, dst):
    # Wikimedia rejects requests without a User-Agent.
    req = urllib.request.Request(url, headers={"User-Agent": "whisperx-demo/1.0"})
    with urllib.request.urlopen(req, timeout=60) as r, open(dst, "wb") as f:
        shutil.copyfileobj(r, f)

def _to_wav(src, dst, start=None, dur=None):
    # Transcode (and optionally trim) to 16 kHz mono WAV -- WhisperX's native rate.
    cmd = [_ffmpeg_exe(), "-y"]
    if start is not None:
        cmd += ["-ss", str(start)]
    cmd += ["-i", src]
    if dur is not None:
        cmd += ["-t", str(dur)]
    cmd += ["-ac", "1", "-ar", "16000", dst]
    p = subprocess.run(cmd, capture_output=True)
    if p.returncode != 0:
        raise RuntimeError("ffmpeg failed:\n" + p.stderr.decode("utf-8", "ignore"))

warmup_clip = "/tmp/whisperx_warmup.wav"
demo_clip   = "/tmp/whisperx_demo.wav"

source_audio = LOCAL_AUDIO
if not source_audio:
    source_audio = "/tmp/apollo11.ogg"
    try:
        _download(APOLLO_URL, source_audio)
        print("Downloaded Apollo 11 landing audio (Wikimedia Commons, PD-USGov-NASA).")
    except Exception as e:
        print("Primary download failed, using fallback ('one small step'):", e)
        _download(APOLLO_FALLBACK_URL, source_audio)

_to_wav(source_audio, demo_clip, APOLLO_START_SEC, APOLLO_DUR_SEC)   # demo clip (full or trimmed)
_to_wav(source_audio, warmup_clip, 0, 15)                            # first 15s = warm-up

print("Demo clip:   ", demo_clip)
print("Warm-up clip:", warmup_clip)

from IPython.display import Audio, display, HTML
display(Audio(demo_clip))

In [ ]:
# ---- Manual multipart/form-data encoder -------------------------------------
# Matches the container's POST /invocations contract exactly:
#   - the audio part MUST be named "file" with a .wav filename
#   - every other field is a plain form-data text part
#   - line endings MUST be CRLF
BOUNDARY = "----whisperxdemoboundary7MA4YWxkTrZu0gW"

def build_multipart(audio_bytes, filename, fields):
    # Returns (body_bytes, content_type_header).
    CRLF = "\r\n"

    # File part: headers, a blank line, the raw bytes, then a trailing CRLF.
    head = CRLF.join([
        "--" + BOUNDARY,
        'Content-Disposition: form-data; name="file"; filename="' + filename + '"',
        "Content-Type: audio/wav",
        "",
    ])
    body = head.encode("utf-8") + CRLF.encode("utf-8") + audio_bytes + CRLF.encode("utf-8")

    # One text part per extra field, then the closing boundary.
    tail = []
    for name, value in fields.items():
        tail.append("--" + BOUNDARY)
        tail.append('Content-Disposition: form-data; name="' + name + '"')
        tail.append("")
        tail.append(str(value))
    tail.append("--" + BOUNDARY + "--")
    tail.append("")
    body += CRLF.join(tail).encode("utf-8")

    content_type = "multipart/form-data; boundary=" + BOUNDARY
    return body, content_type


# srt / vtt / text responses are plain text, not JSON -- do NOT json.loads them.
RAW_FORMATS = {"srt", "vtt", "text"}

def invoke(audio_path, fields, retries=5, sleep_s=30):
    # Reads the file, builds the multipart body, calls the endpoint, and retries
    # the slow first call. Returns (result, seconds): result is a dict for
    # json / verbose_json, or a raw str for srt / vtt / text.
    with open(audio_path, "rb") as f:
        audio_bytes = f.read()
    filename = os.path.basename(audio_path)
    if not filename.endswith(".wav"):
        filename += ".wav"

    body, content_type = build_multipart(audio_bytes, filename, fields)
    raw = fields.get("response_format") in RAW_FORMATS

    for attempt in range(1, retries + 1):
        start = time.time()
        try:
            resp = smr.invoke_endpoint(
                EndpointName=ENDPOINT_NAME,
                ContentType=content_type,
                Accept="application/json",
                Body=body,
            )
            payload = resp["Body"].read()
            secs = time.time() - start
            return (payload.decode("utf-8") if raw else json.loads(payload)), secs
        except (ClientError, ReadTimeoutError) as e:
            # The 60s real-time cap trips while large-v2 + the aligner download on
            # the very first call. Sleep and retry -- this is expected once.
            if attempt == retries:
                raise
            print(f"  attempt {attempt} hit the 60s cap ({type(e).__name__}); "
                  f"waiting {sleep_s}s then retrying -- first call downloads the model ...")
            time.sleep(sleep_s)

## 3. Warm up the endpoint -- run this before your first timed request

The **first** call makes the container download `large-v2` **and** the English wav2vec2 aligner. That can take a few minutes and **several retries** (the 60s real-time cap trips while it downloads -- `invoke()` retries automatically). After the warm-up, calls are fast.

In [ ]:
warmup_result, warmup_secs = invoke(warmup_clip, {"language": "en", "response_format": "json"})
print(f"Warm-up completed in {warmup_secs:.1f}s.")
print("Transcript length:", len(warmup_result.get("text", "")), "chars")

## 4. Transcription

The endpoint is warm. We transcribe the 2-speaker clip with **word-level timestamps** (`timestamp_granularities[]=word` -> forced alignment) and **diarization** (`diarize=true` -> speaker labels), and measure wall-clock throughput.

In [ ]:
result, secs = invoke(demo_clip, {
    "language": "en",
    "response_format": "verbose_json",
    "timestamp_granularities[]": "word",   # literal trailing [] -> word-level timestamps + forced alignment
    "diarize": "true",                      # speaker labels
    "min_speakers": "2",     # <- force at least 2
    "max_speakers": "3",
})
print(f"Transcribed + word-aligned + diarized in {secs:.1f}s.")

In [ ]:
from IPython.display import HTML, display

# Accessible, dark-mode-friendly palette; each speaker gets a stable color.
PALETTE = ["#2563eb", "#dc2626", "#059669", "#7c3aed", "#d97706", "#0891b2"]
DEFAULT_COLOR = "#94a3b8"

def mmss(t):
    t = int(round(float(t)))
    return f"{t // 60:02d}:{t % 60:02d}"

duration = float(result.get("duration", 0.0))
speakers = result.get("speakers", [])
rt = (duration / secs) if secs > 0 else 0.0
color_of = {spk: PALETTE[i % len(PALETTE)] for i, spk in enumerate(speakers)}

banner = (f"{duration:.1f}s audio &middot; {len(speakers)} speakers &middot; "
          f"transcribed + word-aligned + diarized in {secs:.1f}s &rarr; "
          f"<b>{rt:.1f}x realtime</b>")

rows = []
for seg in result.get("segments", []):
    spk = seg.get("speaker", "SPEAKER_??")
    color = color_of.get(spk, DEFAULT_COLOR)
    ts = mmss(seg.get("start", 0.0))
    text = (seg.get("text", "") or "").strip()
    rows.append(
        f'<div style="padding:6px 10px;margin:3px 0;border-radius:6px;'
        f'background:{color}22;border-left:4px solid {color};">'
        f'<span style="font-family:monospace;color:#94a3b8;">[{ts}]</span> '
        f'<span style="color:{color};font-weight:600;">{spk}</span>: '
        f'<span style="color:#e5e7eb;">{text}</span></div>'
    )

html = (
    '<div style="background:#0f172a;padding:16px;border-radius:10px;'
    'font-family:system-ui,-apple-system,sans-serif;max-width:900px;">'
    f'<div style="color:#f1f5f9;font-size:15px;margin-bottom:12px;">{banner}</div>'
    + "".join(rows)
    + '</div>'
)
display(HTML(html))

print(f"{len(result.get('words', []))} word-level timestamps")
print("Speakers:", speakers)

## 5. Bonus -- broadcast-ready captions

The same endpoint emits **SRT** subtitles with highlighted words, ready to drop into a video editor.

In [ ]:
# response_format="srt" returns plain text, not JSON; invoke() handles that.
srt, _ = invoke(demo_clip, {
    "response_format": "srt",
    "highlight_words": "true",
    "max_line_width": "42",
})
srt_path = "/tmp/whisperx_demo.srt"
with open(srt_path, "w") as f:
    f.write(srt)
print("Wrote", srt_path)
print("---- first lines ----")
print("\n".join(srt.splitlines()[:15]))

## 6. Teardown -- **delete the endpoint to stop billing**

> ### 💸 The GPU endpoint bills continuously until deleted. Run this cell as soon as you're done.

In [ ]:
# Each delete is best-effort so a partial deploy still cleans up fully.
for label, fn in [
    ("endpoint",        lambda: sm.delete_endpoint(EndpointName=ENDPOINT_NAME)),
    ("endpoint-config", lambda: sm.delete_endpoint_config(EndpointConfigName=CONFIG_NAME)),
    ("model",           lambda: sm.delete_model(ModelName=MODEL_NAME)),
]:
    try:
        fn()
        print("Deleted", label)
    except Exception as e:
        print(f"Could not delete {label}: {e}")
print("Teardown done.")